# Stage 2 — No GAN, With Drift
### GPVS-Faults | Journal extension of ICSPCS 2024 & ITNAC 2026

This notebook builds directly on `stage1.ipynb`. All model-training code
(`to_tensors`, `load_scenario`, `run_scenario`, the CNN-LSTM training loop) is
**byte-identical to Stage 1** — the only change is what data feeds into it.

Stage 2 = Stage 1's data pipeline + **Phase 3 (Vdc detrending)** +
**Phase 4 (parametric monotonic drift injection)** from
`concept_drift_gan_methodology.md`, then the same two classifiers
(LSTM-XGB, CNN-LSTM-v2) re-evaluated on the drifted data.

**Design decisions made in this notebook (flagged for review — see
`drift_injection.py` docstring for full rationale):**

1. **Detrending scope**: Vdc detrended for Normal (class 0) rows only, across
   its own train+val+test window. This is the only Normal data in
   `base_splits.pkl` — a much narrower slice than the full raw corpus used for
   the Phase 2 EDA audit, so the local trend removed here is modest.
2. **Drift timeline**: a **local, per-class** artificial progress variable
   τ ∈ [0,1] (0 at that class's first train row, 1 at its last test row),
   rather than a single global real-time timeline. Each class occupies a
   narrow, non-overlapping slice of the ~11s recording, so a global-time ramp
   would leave almost no τ variation *within* any one class's split — it would
   break exactly the chunk-based manipulation check Phase 4.4 asks for.
3. **Injection scope**: applied to **all 8 classes**, not Normal-only, since
   the drift represents a shared physical degradation process (panel/inverter
   aging) that would affect sensor readings regardless of which fault happens
   to be concurrently present.
4. **Calibration metric**: severity for Ipv/Vpv/Iabc is matched to
   **3–5× the Vdc floor in z-score units** (using the frozen Stage-1 scaler's
   per-feature std), not raw units. Ipv/Iabc's natural raw scale is tiny
   relative to Vdc's, so matching the roadmap's raw-volts target literally
   would demand physically absurd shifts. Matching effect size in the same
   standardized space the classifiers actually see is the more defensible
   reading of "3–5× more severe than the natural floor."


In [1]:
import pandas as pd, numpy as np, os, sys, torch, torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
sys.path.insert(0, "..")  # ensure project root is in the path
from base_splits import build_base_splits, load_splits, save_splits, summarize_splits, load_dataframe
from drift_injection import (detrend_vdc_normal, inject_drift, calibrate_amplitude,
                              manipulation_check, DRIFT_FEATURES,
                              VDC_FLOOR_W_RAW, VDC_FLOOR_KS_RAW)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix)
from scipy.stats import ks_2samp, wasserstein_distance
from models.lstm_xgb import LSTM_XGB
from models.cnn_lstm import CNN_LSTM_v1, CNN_LSTM_v2
from utils import reset_gpu_peak_memory, get_gpu_peak_memory_mb, measure_inference_time, Timer, count_parameters

splits_raw = load_splits(path="../base_splits.pkl")
print(summarize_splits(splits_raw))

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

cols = list(splits_raw[0].train.columns[1:-1])
print("feature cols:", cols)


Loaded base splits for 8 classes from /home/maddie/cd-study/Stages/base_splits.pkl
   label  train_n  val_n  test_n  time_start   time_end
0      0     1000    300     300    3.993639   4.153525
1      1     1000    300     300    9.156422   9.316307
2      2     1000    300     300    4.507887   4.667771
3      3     1000    300     300    4.866492   5.026376
4      4     1000    300     300    1.528859   1.688743
5      5     1000    300     300    8.582310   8.742194
6      6     1000    300     300    8.765543   8.925427
7      7     1000    300     300   10.680525  10.840409
device: NVIDIA A30
feature cols: ['Ipv', 'Vpv', 'Vdc', 'ia', 'ib', 'ic', 'va', 'vb', 'vc', 'Iabc', 'If', 'Vabc', 'Vf']


> **Execution note:** this sandbox has no `torch`/`xgboost`/GPU, so cells
> that import them are included as ready-to-run code but were not executed
> here. Everything else in this notebook (Phase 3 detrending, Phase 4
> injection + calibration, the manipulation check, and the frozen-scaler
> z-scoring) **was actually executed**, against the real `base_splits.pkl` you
> uploaded — the printed outputs below are genuine, not illustrative.

## Phase 3 — Vdc Detrending (Normal class only)

Fit a linear trend to Vdc across the Normal partition's own train→val→test
window (verified contiguous in `Time`), subtract it, and re-center on the
original mean. Fault-class rows are asserted untouched.


In [2]:
splits_detrended, vdc_trend_info = detrend_vdc_normal(splits_raw, normal_label=0)

# sanity: fault-class rows must be bit-identical to the raw data
for lbl in range(1, 8):
    assert (splits_detrended[lbl].train["Vdc"].to_numpy()
            == splits_raw[lbl].train["Vdc"].to_numpy()).all()
print("Fault-class Vdc rows confirmed untouched by detrending.")

# Phase 3.4 verification: local Normal-window Vdc KS/W should drop after detrending
# (this is the LOCAL 1600-row window in base_splits.pkl, not the full EDA corpus
# where the 0.553 / 1.089 floor was originally measured -- see markdown above)
for tag, s in [("raw", splits_raw), ("detrended", splits_detrended)]:
    tr, te = s[0].train["Vdc"].to_numpy(), s[0].test["Vdc"].to_numpy()
    print(f"Normal Vdc train-vs-test [{tag:>10}]  "
          f"KS={ks_2samp(tr, te).statistic:.4f}  W={wasserstein_distance(tr, te):.4f}")


Fault-class Vdc rows confirmed untouched by detrending.
Normal Vdc train-vs-test [       raw]  KS=0.3807  W=0.1609
Normal Vdc train-vs-test [ detrended]  KS=0.1550  W=0.0938


## Phase 4 — Drift Injection Design & Calibration

Target features: **Ipv, Vpv, Iabc** (the moderate-response group informed by
F2/F4 — see Phase 4.1). Injection: `value - amplitude * tau`, τ local to each
class (Phase 4.2/4.3: parametric, monotonic, gradual ramp).

Calibration target: **4× the Vdc natural-drift floor**, expressed in z-score
units via the frozen scaler's per-feature std (see design note #4 above), then
converted back to each feature's own raw units for injection.


In [3]:
scaler_ref = StandardScaler().fit(splits_raw[0].train[cols])  # read-only, for sigma
sigma = dict(zip(cols, scaler_ref.scale_))

TARGET_SEVERITY_X = 4.0   # roadmap: 3-5x the Vdc floor; midpoint as the primary operating point
floor_z = VDC_FLOOR_W_RAW / sigma["Vdc"]
target_z = TARGET_SEVERITY_X * floor_z
print(f"Vdc natural-drift floor (Phase 2 EDA, full corpus): KS={VDC_FLOOR_KS_RAW}  W={VDC_FLOOR_W_RAW}")
print(f"floor in z-score units: {floor_z:.4f}  ->  target ({TARGET_SEVERITY_X}x): {target_z:.4f}\n")

amplitudes, achieved = {}, {}
for feat in DRIFT_FEATURES:
    target_raw = target_z * sigma[feat]
    amp, w = calibrate_amplitude(splits_detrended, feat, target_raw, scope="all")
    amplitudes[feat] = amp
    achieved[feat] = w
    print(f"  {feat:5s}  sigma={sigma[feat]:.4f}  target_raw_W={target_raw:.5f}  "
          f"amplitude={amp:.5f}  achieved_mean_W={w:.5f}")


Vdc natural-drift floor (Phase 2 EDA, full corpus): KS=0.553  W=1.089
floor in z-score units: 1.7346  ->  target (4.0x): 6.9384

  Ipv    sigma=0.0991  target_raw_W=0.68754  amplitude=1.18171  achieved_mean_W=0.69683
  Vpv    sigma=0.2920  target_raw_W=2.02635  amplitude=3.40364  achieved_mean_W=2.02585
  Iabc   sigma=0.0031  target_raw_W=0.02163  amplitude=0.03516  achieved_mean_W=0.02146


In [4]:
splits_stage2, tau_lookup = inject_drift(splits_detrended, amplitudes,
                                          features=DRIFT_FEATURES, scope="all")
print("Injected drift into:", DRIFT_FEATURES, " | scope: all 8 classes")


Injected drift into: ['Ipv', 'Vpv', 'Iabc']  | scope: all 8 classes


## Phase 4.4 — Manipulation Check

KS + Wasserstein (train vs. test, mean across all 8 classes), before vs. after
injection, across **all 13 features** — confirming the targeted features moved
by the intended amount and nothing else drifted as a side effect.


In [5]:
mc = manipulation_check(splits_detrended, splits_stage2, cols)
summary = (mc.pivot_table(index="feature", columns="condition",
                          values=["ks", "wasserstein"], aggfunc="mean")
             .loc[cols])
pd.set_option("display.width", 120)
print(summary.round(4).to_string())


               ks         wasserstein         
condition   after  before       after   before
feature                                       
Ipv        0.9831  0.1880      0.6968   0.0219
Vpv        0.9481  0.1674      2.0258   0.1273
Vdc        0.2609  0.2609      0.2606   0.2606
ia         0.1154  0.1154      0.2190   0.2190
ib         0.1268  0.1268      0.2358   0.2358
ic         0.0720  0.0720      0.0563   0.0563
va         0.1052  0.1052     24.8762  24.8762
vb         0.1025  0.1025     23.6625  23.6625
vc         0.0710  0.0710     16.1277  16.1277
Iabc       0.9800  0.9115      0.0215   0.0123
If         0.4324  0.4324      0.0674   0.0674
Vabc       0.8037  0.8037      0.1591   0.1591
Vf         0.4446  0.4446      0.0043   0.0043


**Reading the table:**
- **Ipv, Vpv** — clean, large, uniform jump in both KS and Wasserstein — the
  drift landed as designed.
- **Iabc** — KS is already ~0.91 *before* injection and barely moves further
  (→0.98); Wasserstein does show the expected multiplicative jump (~0.012 →
  ~0.021) but stays numerically tiny. Iabc's natural std is extremely small
  (≈0.003) relative to its mean, so its train/test value ranges are already
  almost disjoint at baseline — KS saturates near 1 from natural fine-grained
  separation alone and isn't an informative severity signal for this feature.
  **Wasserstein is the metric to report for Iabc**; KS is fine for Ipv/Vpv.
- **Vdc, ia/ib/ic, va/vb/vc, If, Vabc, Vf** — identical before/after, confirming
  injection didn't leak into non-targeted features.

## Frozen Scaler + Z-Scaling

Same frozen-scaler discipline as Stage 1: **fit once on Stage 1's raw,
undrifted, undetrended** Normal-class training data, then reused via
`.transform()` only — never refit. Refitting on the Stage 2 data would
re-center the injected drift out of existence.


In [6]:
# Frozen scaler discipline: fit ONCE on Stage 1's raw, undrifted, undetrended
# Normal-class training data -- identical fit to stage1.ipynb. NEVER refit here;
# refitting on drifted data would re-center the injected drift out of existence.
scaler = StandardScaler()
scaler.fit(splits_raw[0].train[cols])

dct = dict()
for i in range(len(splits_stage2)):
    dct[i] = dict()
    dct[i].update({
        "train": pd.DataFrame(scaler.transform(splits_stage2[i].train[cols]), columns=cols,
                              index=splits_stage2[i].train.index).assign(Fault=i),
        "val": pd.DataFrame(scaler.transform(splits_stage2[i].val[cols]), columns=cols,
                            index=splits_stage2[i].val.index).assign(Fault=i),
        "test": pd.DataFrame(scaler.transform(splits_stage2[i].test[cols]), columns=cols,
                             index=splits_stage2[i].test.index).assign(Fault=i),
    })
print("Stage 2 dct built:", {i: {k: len(v) for k, v in dct[i].items()} for i in dct})


Stage 2 dct built: {0: {'train': 1000, 'val': 300, 'test': 300}, 1: {'train': 1000, 'val': 300, 'test': 300}, 2: {'train': 1000, 'val': 300, 'test': 300}, 3: {'train': 1000, 'val': 300, 'test': 300}, 4: {'train': 1000, 'val': 300, 'test': 300}, 5: {'train': 1000, 'val': 300, 'test': 300}, 6: {'train': 1000, 'val': 300, 'test': 300}, 7: {'train': 1000, 'val': 300, 'test': 300}}


## Model Evaluation — LSTM-XGB and CNN-LSTM-v2 on Stage 2 Data

Everything below is **unchanged from `stage1.ipynb`** — same `to_tensors`,
`load_scenario`, `run_scenario`, and CNN-LSTM training loop, only pointed at
the Stage 2 `dct` built above. Requires `torch`/`xgboost`/GPU, so it's included
as ready-to-run code but not executed in this sandbox.


In [7]:
def to_tensors(df, cols):
    """Convert a (features + Fault) DataFrame into model-ready tensors.
    X: (N, 1, len(cols)) so the LSTM sees the len(cols) features as a
       length-len(cols) sequence with 1 channel each (matches LSTM_XGB's
       expected input shape).
    y: (N,) integer Fault labels.
    """
    X = torch.from_numpy(df[cols].to_numpy(dtype="float32")).unsqueeze(1)
    y = torch.from_numpy(df["Fault"].to_numpy(dtype="int64"))
    return X, y


def load_scenario(dct, cols):
    """Build combined 8-class train/val/test tensors directly from the
    in-memory `dct` dict (built in the Z-scale cell), instead of reading
    per-scenario CSVs off disk.

    dct is keyed by class label: dct[i]["train"/"val"/"test"] is a
    per-class DataFrame of z-scored features + a Fault column. We
    concatenate across classes to get the full multiclass split.
    """
    train_df = pd.concat([dct[i]["train"] for i in sorted(dct)], axis=0)
    val_df   = pd.concat([dct[i]["val"]   for i in sorted(dct)], axis=0)
    test_df  = pd.concat([dct[i]["test"]  for i in sorted(dct)], axis=0)
    return (to_tensors(train_df, cols),
            to_tensors(val_df,   cols),
            to_tensors(test_df,  cols))


def run_scenario(scenario_idx, dct, cols, device,
                 epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(dct, cols)

    model = LSTM_XGB(n_classes=8, lstm_hidden=32, device=device, seed=seed)
    n_params, params_by_type = count_parameters(model.backbone)  # LSTM only
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} (LSTM_XGB) ===")
        print(f"  LSTM params: {n_params:,}  breakdown: {params_by_type}")

    # ---- Stage 1: LSTM training ----
    with Timer(device) as stage1_timer:
        model.fit_lstm(X_tr, y_tr, X_val=X_va, y_val=y_va,
                       epochs=epochs, lr=lr, weight_decay=weight_decay,
                       batch_size=50, seed=seed, verbose=verbose)

    # ---- Stage 2: XGBoost fitting ----
    with Timer(device=None) as stage2_timer:   # XGBoost is CPU-bound
        model.fit_xgb(X_tr, y_tr)

    train_sec_total = stage1_timer.elapsed + stage2_timer.elapsed
    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- Inference timing ----
    X_te_dev = X_te.to(device)
    inf_stats = measure_inference_time(model.predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- Test accuracy ----
    y_true = y_te.numpy()
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: stage1(LSTM)={stage1_timer.elapsed:.1f}s  "
              f"stage2(XGB)={stage2_timer.elapsed:.1f}s  "
              f"total={train_sec_total:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample")

    return {
        "scenario": scenario_idx, "model": "LSTM_XGB",
        "n_train": len(X_tr),
        "accuracy": acc, "precision": p, "recall": r, "f1": f,
        "confusion": cm,
        "n_params": n_params,
        "train_sec": round(train_sec_total, 2),
        "train_sec_stage1": round(stage1_timer.elapsed, 2),
        "train_sec_stage2": round(stage2_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }


## Multi-Seed Evaluation (LSTM-XGB)

Repeats LSTM-XGB training across model-training seeds 0-4, holding the data
split fixed (`DATA_SPLIT_SEED = 20260827`, unchanged across seeds and
stages). This is a **repeated-restart / seed-variance study**, not Monte
Carlo cross-validation -- the train/val/test partition itself never
changes, only the model's own stochastic initialization/shuffling
(LSTM weight init, minibatch order, XGBoost's `random_state`). That's
deliberate: it isolates model-training variance from data-sampling
variance, unlike Li et al.'s protocol, where resampling across runs
conflates the two. It also means results are paired across seeds and
across stages, since every seed sees the exact same held-out rows.

The single seed=0 run above is kept as-is for continuity with prior
discussion; this section adds the full 5-seed picture around it.


In [9]:
from multiseed import run_multi_seed, aggregate_results, summary_table, per_class_accuracy, DEFAULT_SEEDS

print(f"LSTM-XGB across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
lstm_xgb_seed_results = run_multi_seed(
    run_scenario, seeds=DEFAULT_SEEDS, scenario_idx=0, dct=dct, cols=cols, device=device)

lstm_xgb_agg = aggregate_results(lstm_xgb_seed_results)
print("\nLSTM-XGB, mean +/- std across seeds:")
for m, s in lstm_xgb_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}   (min={s['min']:.4f} max={s['max']:.4f})")


LSTM-XGB across 5 seeds: [0, 1, 2, 3, 4]
  seed=0  acc=0.7146  P=0.7240  R=0.7146  F1=0.6804  train_sec=53.6
  seed=1  acc=0.8167  P=0.7727  R=0.8167  F1=0.7716  train_sec=51.1
  seed=2  acc=0.7675  P=0.7332  R=0.7675  F1=0.7071  train_sec=51.4
  seed=3  acc=0.8342  P=0.9252  R=0.8342  F1=0.8048  train_sec=49.1
  seed=4  acc=0.7758  P=0.8393  R=0.7758  F1=0.7511  train_sec=45.8

LSTM-XGB, mean +/- std across seeds:
  accuracy    0.7817 +/- 0.0467   (min=0.7146 max=0.8342)
  precision   0.7989 +/- 0.0839   (min=0.7240 max=0.9252)
  recall      0.7817 +/- 0.0467   (min=0.7146 max=0.8342)
  f1          0.7430 +/- 0.0498   (min=0.6804 max=0.8048)


In [10]:
# CNN-LSTM training pipeline -- byte-identical to stage1.ipynb. to_tensors()
# and load_scenario() are reused as-is from the previous cell.

def make_model(model_cls, device, seed=0, **kwargs):
    """Fresh CNN-LSTM with Xavier init for Conv/Linear; default PyTorch init
    for LSTM (MATLAB's Glorot applies to Conv and FC, LSTM uses its own)."""
    torch.manual_seed(seed)
    model = model_cls(n_classes=8, **kwargs).to(device)
    for m in model.modules():
        if isinstance(m, nn.Conv1d):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=0):
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(X_tr), generator=g)
    X_tr_s, y_tr_s = X_tr[perm], y_tr[perm]
    train_loader = DataLoader(TensorDataset(X_tr_s, y_tr_s),
                              batch_size=batch_size, shuffle=False)
    val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size)
    return train_loader, val_loader, test_loader


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate_loader(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss    += criterion(logits, yb).item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for xb, yb in loader:
        logits = model(xb.to(device))
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(yb.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)


def run_scenario_cnn_lstm(scenario_idx, dct, cols, model_cls, device,
                          epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(dct, cols)
    train_loader, val_loader, test_loader = make_loaders(
        X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=seed)

    model = make_model(model_cls, device, seed=seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr,
                           betas=(0.9, 0.999), eps=1e-8,
                           weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    n_params, params_by_type = count_parameters(model)
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} ({model_cls.__name__}) ===")
        print(f"  params: {n_params:,}  breakdown: {params_by_type}")

    with Timer(device) as train_timer:
        for ep in range(1, epochs + 1):
            tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                              criterion, optimizer, device)
            va_loss, va_acc = evaluate_loader(model, val_loader,
                                              criterion, device)
            scheduler.step()
            if verbose and (ep == 1 or ep % 10 == 0 or ep == epochs):
                print(f"  ep {ep:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f}"
                      f" | val loss {va_loss:.4f} acc {va_acc:.3f}")

    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- Inference timing ----
    X_te_dev = X_te.to(device)
    def _predict(X):
        model.eval()
        with torch.no_grad():
            return model(X).argmax(1)
    inf_stats = measure_inference_time(_predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- Test accuracy ----
    y_true, y_pred = predict_all(model, test_loader, device)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: train={train_timer.elapsed:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample  "
              f"peak_mem={peak_mem_mb:.1f}MB")

    return {
        "scenario":    scenario_idx,
        "model":       model_cls.__name__,
        "n_train":     len(X_tr),
        "accuracy":    acc,
        "precision":   p,
        "recall":      r,
        "f1":          f,
        "confusion":   cm,
        "n_params":    n_params,
        "train_sec":   round(train_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }


## Multi-Seed Evaluation (CNN-LSTM-v2)

Same 5 seeds, same fixed data split. The single seed=0 run above is kept
for continuity; this adds the full picture.


In [ ]:
MODEL_CLS = CNN_LSTM_v2  # or CNN_LSTM_v1

print(f"CNN-LSTM-v2 across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
cnn_lstm_seed_results = run_multi_seed(
    run_scenario_cnn_lstm, seeds=DEFAULT_SEEDS, scenario_idx=1, dct=dct, cols=cols,
    model_cls=MODEL_CLS, device=device)

cnn_lstm_agg = aggregate_results(cnn_lstm_seed_results)
print("\nCNN-LSTM-v2, mean +/- std across seeds:")
for m, s in cnn_lstm_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}   (min={s['min']:.4f} max={s['max']:.4f})")


CNN-LSTM-v2 across 5 seeds: [0, 1, 2, 3, 4]
  seed=0  acc=0.8750  P=0.8119  R=0.8750  F1=0.8331  train_sec=68.6
  seed=1  acc=0.8750  P=0.8122  R=0.8750  F1=0.8332  train_sec=65.9
  seed=2  acc=0.8746  P=0.8002  R=0.8746  F1=0.8284  train_sec=66.2
  seed=3  acc=0.8708  P=0.9147  R=0.8708  F1=0.8317  train_sec=66.2
  seed=4  acc=0.8750  P=0.8092  R=0.8750  F1=0.8321  train_sec=65.5

CNN-LSTM-v2, mean +/- std across seeds:
  accuracy    0.8741 +/- 0.0018   (min=0.8708 max=0.8750)
  precision   0.8296 +/- 0.0478   (min=0.8002 max=0.9147)
  recall      0.8741 +/- 0.0018   (min=0.8708 max=0.8750)
  f1          0.8317 +/- 0.0020   (min=0.8284 max=0.8332)


## Combined Multi-Seed Summary

Mean +/- std across the 5 seeds for both models, plus per-class accuracy
averaged across seeds (diagonal of the seed-averaged, row-normalized
confusion matrix). Results are pickled for the eventual Stage-1-vs-Stage-2
(and later Stage-3/4) paired comparison via `multiseed.paired_diff`.


In [13]:
combined = summary_table({"LSTM-XGB": lstm_xgb_agg, "CNN-LSTM-v2": cnn_lstm_agg})
print(combined.round(4).to_string(index=False))

class_names = ["Normal", "F1", "F2", "F3", "F4", "F5", "F6", "F7"]
pc = pd.DataFrame({
    "LSTM-XGB": per_class_accuracy(lstm_xgb_agg["mean_confusion_rate"], class_names),
    "CNN-LSTM-v2": per_class_accuracy(cnn_lstm_agg["mean_confusion_rate"], class_names),
})
print("\nPer-class accuracy (mean confusion diagonal across seeds):")
print(pc.round(4).to_string())

import pickle
with open("stage2_seed_results.pkl", "wb") as f:
    pickle.dump({"lstm_xgb": lstm_xgb_agg, "cnn_lstm_v2": cnn_lstm_agg}, f)
print("\nSaved stage2_seed_results.pkl")


      model  n_seeds  accuracy_mean  accuracy_std  precision_mean  precision_std  recall_mean  recall_std  f1_mean  f1_std
   LSTM-XGB        5         0.7817        0.0467          0.7989         0.0839       0.7817      0.0467   0.7430  0.0498
CNN-LSTM-v2        5         0.8741        0.0018          0.8296         0.0478       0.8741      0.0018   0.8317  0.0020

Per-class accuracy (mean confusion diagonal across seeds):
        LSTM-XGB  CNN-LSTM-v2
Normal    0.3707       0.9820
F1        0.9927       1.0000
F2        1.0000       1.0000
F3        0.0320       0.0107
F4        0.9747       1.0000
F5        1.0000       1.0000
F6        0.9140       1.0000
F7        0.9700       1.0000

Saved stage2_seed_results.pkl


## Next Steps

- Run this notebook end-to-end in your GPU environment to get Stage 2's actual
  accuracy/F1 and compare against Stage 1's ~99.4% (LSTM-XGB) / 100%
  (CNN-LSTM-v2). Per-class breakdown matters most for F2/F4 (directly
  drift-targeted) and F6/F7 (already borderline in Stage 1).
- If the injected severity (currently 4× the Vdc floor) turns out to barely
  move accuracy, or moves it to floor immediately, `TARGET_SEVERITY_X` is a
  single knob to sweep (3–5x per the roadmap) before locking in Stage 2's
  reported operating point.
- Stage 3 (GAN, no drift) can reuse `dct`-building pattern from Stage 1
  unchanged; Stage 4 (GAN + drift) can reuse `drift_injection.py` as-is,
  swapping in the drift-severity sub-matrix from roadmap §5.2.
- Consider running all 5 model-training seeds here too, for the same paired
  comparison against Stage 1 you're planning to backfill there.
